# This is a placeholder neural network to make sure the NN can ingest the data.
... it's not actually learning anything meaningful yet

In [1]:
import torch
from torch import nn
from torch_geometric.nn import GCNConv, global_mean_pool

print(torch.__version__)

C:\Users\jackd\source\colmap-experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.9.0+cu128


In [2]:
import pycolmap
import pathlib
import utils.subset

# Load the Colmap data into a torch dataset
model_path = pathlib.Path('./output/01_nerf/02_lego_large/1762043798762/0')
reconstruction = pycolmap.Reconstruction(model_path)

graph_data = utils.subset.reconstruction_to_pyg_data(reconstruction)
graph_data

Data(x=[69, 3], edge_index=[2, 3686])

In [3]:
graph_data.num_features

3

In [5]:
# This is a super simple network that takes a graph as input, and outputs a single scalar
class DummyGCN(nn.Module):
    def __init__(self, node_feature_size: int):
        super().__init__()
        self.conv1 = GCNConv(node_feature_size, 4)
        self.linear = nn.Linear(4,1)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = h.mean(dim=0) # simple pooling. no batching
        h = self.linear(h)
        return h

In [6]:
model = DummyGCN(graph_data.num_features)
print(model)

DummyGCN(
  (conv1): GCNConv(3, 4)
  (linear): Linear(in_features=4, out_features=1, bias=True)
)


In [7]:
h = model(graph_data.x, graph_data.edge_index)

In [8]:
h.detach()

tensor([0.1380])

In [10]:
# Train the network
model = DummyGCN(graph_data.num_features)
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)
criterion = nn.MSELoss()
dummy_target = torch.tensor([42.0])
for epoch in range(10):
    optimizer.zero_grad()
    out = model(graph_data.x, graph_data.edge_index)
    loss = criterion(out, dummy_target) # dummy hardcoded value
    # compute gradient
    loss.backward()
    # update weights
    optimizer.step()

In [11]:
# Example prediction
model(graph_data.x, graph_data.edge_index)

tensor([42.2506], grad_fn=<ViewBackward0>)

Success!! Data goes in, and we learn out hard coded number 42. Next, we will try to learn something legit.

# Next, let's try taking in two Graphs

In [14]:
print(graph_data.batch)

None


In [40]:
NUM_NODE_FEATURES = graph_data.num_features
import torch.nn.functional as F
import torch_geometric.data

# This is a super simple network that takes a graph as input, and outputs a single scalar
class GraphEmbeddingGCN_v1(nn.Module):
    def __init__(self, num_node_features: int, output_dim: int):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, 4)
        self.conv2 = GCNConv(4, 4)
        self.linear = nn.Linear(4, output_dim)

    def forward(self, graph_data: torch_geometric.data.Data):
        h = self.conv1(graph_data.x, graph_data.edge_index)
        h = F.relu(h)
        h = self.conv2(h, graph_data.edge_index)
        h = F.relu(h)
        # Pool all the node features to get a graph-level representation
        h = global_mean_pool(h, graph_data.batch)
        h = self.linear(h)
        return h

# This is a super simple network that takes a graph as input, and outputs a single scalar
class SiameseGCN_v1(nn.Module):
    def __init__(self, num_node_features: int):
        super().__init__()
        # Create twin branches that process each graph in tandem
        graph_embedding_dim = 4
        self.sisterA = GraphEmbeddingGCN_v1(num_node_features, output_dim = graph_embedding_dim)
        self.sisterB = GraphEmbeddingGCN_v1(num_node_features, output_dim = graph_embedding_dim)

        # This next part will take as input the embeddings from the sister network
        # self.conv1 = GCNConv(graph_embedding_dim * 2, 4)
        self.linear1 = nn.Linear(graph_embedding_dim * 2, 8)
        self.linear2 = nn.Linear(graph_embedding_dim * 2, 8)
        self.mlp = nn.Sequential(
            nn.Linear(graph_embedding_dim * 2, 8),
            nn.ReLU(),
            nn.Linear(8, 8),
            nn.ReLU(),
            # Output 3 scalars for (x, y, z) translation prediction
            nn.Linear(8, 3)
        )

    def forward(self, graph_data_1: torch_geometric.data.Data, graph_data_2: torch_geometric.data.Data):
        # Project graph1 and graph2 into the "embedding space" (idk if this is properly called an embedding space...)
        e1 = self.sisterA(graph_data_1)
        e2 = self.sisterB(graph_data_2)

        # Concatenate the features
        h = torch.cat((e1, e2), dim=1)
        # Run through an MLP to predict 3
        h = self.mlp(h)
        return h

In [47]:
# Example of running the graph embedding bit of the network
temp = GraphEmbeddingGCN_v1(graph_data.num_features, 4)
print(temp)
res = temp(graph_data)
print(res)

GraphEmbeddingGCN_v1(
  (conv1): GCNConv(3, 4)
  (conv2): GCNConv(4, 4)
  (linear): Linear(in_features=4, out_features=4, bias=True)
)
tensor([[ 0.0584,  0.3661, -0.2890, -0.0849]], grad_fn=<AddmmBackward0>)


In [50]:
siam_gcn = SiameseGCN_v1(graph_data.num_features)
print(siam_gcn)

SiameseGCN_v1(
  (sisterA): GraphEmbeddingGCN_v1(
    (conv1): GCNConv(3, 4)
    (conv2): GCNConv(4, 4)
    (linear): Linear(in_features=4, out_features=4, bias=True)
  )
  (sisterB): GraphEmbeddingGCN_v1(
    (conv1): GCNConv(3, 4)
    (conv2): GCNConv(4, 4)
    (linear): Linear(in_features=4, out_features=4, bias=True)
  )
  (linear1): Linear(in_features=8, out_features=8, bias=True)
  (linear2): Linear(in_features=8, out_features=8, bias=True)
  (mlp): Sequential(
    (0): Linear(in_features=8, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=3, bias=True)
  )
)


In [61]:
# Now, let's run two graph networks through it to make sure the sizes all line up
import utils.subset
a_ids, b_ids = utils.subset.sample_image_subsets(reconstruction)

In [89]:
# Convert the reconstruction data into two sub-graphs
graph_a = utils.subset.reconstruction_to_pyg_data(reconstruction, a_ids)
graph_b = utils.subset.reconstruction_to_pyg_data(reconstruction, b_ids)

In [90]:
import utils.vis
# visualize the data
utils.vis.plot_3D_graph(graph_a, graph_b)

In [91]:
# Now, we perturb one of the dataset translations.
perturb_translation = torch.tensor([1.0, 2.0, 3.0])
graph_b.x= graph_b.x + perturb_translation

In [92]:
# Now, let's visualize again, but this time the data for graph_b has been translated
utils.vis.plot_3D_graph(graph_a, graph_b)

In [93]:
# Now let's feed both graphs that into the siameseGCN
res = siam_gcn(graph_a, graph_b)
# Sanity check. we should get 3 scalars
print(res.detach())

tensor([[-0.0381,  0.2355,  0.2655]])


# Train the siamese GCN!

In [100]:
assert(graph_a.num_features == graph_b.num_features)
model = SiameseGCN_v1(graph_a.num_features)
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)
criterion = nn.MSELoss()
# Our target is the value we have perturbed the data by!
# We must unsqueeze to convert the shape from (3) to (1, 3), which matches the output shape of our model
target = perturb_translation.unsqueeze(0)
for epoch in range(500):
    optimizer.zero_grad()
    out = model(graph_a, graph_b)
    loss = criterion(out, target)
    # compute gradient
    loss.backward()
    # update weights
    optimizer.step()

In [101]:
# See our prediction!
prediction = model(graph_a, graph_b)
prediction

tensor([[1.0000, 2.0000, 3.0000]], grad_fn=<AddmmBackward0>)

In [102]:
model.eval()
with torch.no_grad():
    prediction = model(graph_a, graph_b)
    out = prediction.detach()
    print(out)

tensor([[1.0000, 2.0000, 3.0000]])


yay! we learned the hardcoded [1, 2, 3] vector based on two graph inputs! This is progress, although we still haven't really learned anything beyond just memorizing this single vector. Next, we need to pass in several different perturbed data. And, hold out one validation perturbation.